# Introdução aos Decorators do Pandera
Os decorators do Pandera são uma forma elegante de aplicar validação de dados em funções Python. Eles permitem definir esquemas de validação diretamente nos parâmetros de entrada e saída das funções.

# Tipos de Decorators

### 1. @check_input

In [ ]:
import numpy as np
import pandera.pandas as pa
import pandas as pd

# Definindo esquema para validação de entrada
schema_input = pa.DataFrameSchema({
    'idade': pa.Column(int, checks=pa.Check.ge(0)),
    'salario': pa.Column(float, checks=pa.Check.gt(0))
})

@pa.check_input(schema_input)
def calcular_aumento_salarial(df):
    df['novo_salario'] = df['salario'] * 1.1
    return df

# Exemplo de uso
dados = pd.DataFrame({
    'idade': [25, 30, 35],
    'salario': [5000.0, 6000.0, 7000.0]
})

try:
    resultado = calcular_aumento_salarial(dados)
    print(resultado)
except pa.errors.SchemaError as e:
    print(f"Erro na validação: {e}")

### 2. @check_output

In [ ]:
# Definindo esquema para validação de saída
schema_output = pa.DataFrameSchema({
    'idade': pa.Column(int, checks=pa.Check.ge(0)),
    'salario': pa.Column(float, checks=pa.Check.gt(0)),
    'novo_salario': pa.Column(float, checks=pa.Check.gt(0))
})

@pa.check_output(schema_output)
def calcular_aumento_salarial(df):
    df['novo_salario'] = df['salario'] * 1.1
    return df

### 3. @check_io

In [ ]:
# Definindo esquemas para entrada e saída
schema_input = pa.DataFrameSchema({
    'idade': pa.Column(int, checks=pa.Check.ge(0)),
    'salario': pa.Column(float, checks=pa.Check.gt(0))
})

schema_output = pa.DataFrameSchema({
    'idade': pa.Column(int, checks=pa.Check.ge(0)),
    'salario': pa.Column(float, checks=pa.Check.gt(0)),
    'novo_salario': pa.Column(float, checks=pa.Check.gt(0))
})

@pa.check_io(df_in=schema_input, df_out=schema_output)
def calcular_aumento_salarial(df):
    df['novo_salario'] = df['salario'] * 1.1
    return df

# Exemplos Práticos

### 1. Validação em Funções de Transformação

In [ ]:
# Esquema para dados de vendas
schema_vendas = pa.DataFrameSchema({
    'data': pa.Column('datetime64[ns]'),
    'produto': pa.Column(str),
    'quantidade': pa.Column(int, checks=pa.Check.gt(0)),
    'valor': pa.Column(float, checks=pa.Check.gt(0))
})

@pa.check_input(schema_vendas)
def gerar_relatorio_vendas(df):
    # Agrupando por produto
    relatorio = df.groupby('produto').agg({
        'quantidade': 'sum',
        'valor': 'sum'
    }).reset_index()
    
    # Calculando média de valor por unidade
    relatorio['valor_medio'] = relatorio['valor'] / relatorio['quantidade']
    
    return relatorio

### 2. Validação em Funções de Machine Learning

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Esquema para dados de treinamento
schema_treinamento = pa.DataFrameSchema({
    'idade': pa.Column(float, checks=pa.Check.ge(0)),
    'renda': pa.Column(float, checks=pa.Check.gt(0)),
    'escolaridade': pa.Column(str, checks=pa.Check.isin(['fundamental', 'medio', 'superior'])),
    'target': pa.Column(int, checks=pa.Check.isin([0, 1]))
})

@pa.check_input(schema_treinamento)
def treinar_modelo(df):
    # Preparando dados
    X = df.drop('target', axis=1)
    y = df['target']
    
    # Treinando modelo
    modelo = RandomForestClassifier()
    modelo.fit(X, y)
    
    return modelo

### 3. Validação em APIs

In [ ]:

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

# Esquema para dados de usuário
schema_usuario = pa.DataFrameSchema({
    'nome': pa.Column(str, checks=pa.Check.str_length(min=3, max=100)),
    'idade': pa.Column(int, checks=pa.Check.ge(0)),
    'email': pa.Column(str, checks=pa.Check.str_matches(r'^[^@]+@[^@]+\.[^@]+$'))
})

class Usuario(BaseModel):
    nome: str
    idade: int
    email: str

@app.post("/usuarios")
@pa.check_input(schema_usuario)
async def registrar_usuario(usuario: Usuario):
    # Convertendo para DataFrame
    df = pd.DataFrame([usuario.dict()])
    
    # Processando usuário...
    return {"mensagem": "Usuário registrado com sucesso"}

# Boas Práticas para Uso de Decorators
Defina Schemas Reutilizáveis: Crie schemas que possam ser reutilizados em diferentes funções
    
Documente as Regras: Mantenha documentação clara das regras de validação
    
Use Schemas Específicos: Crie schemas específicos para cada tipo de operação
    
Combine com Type Hints: Use type hints junto com os decorators para melhor documentação
    
Trate Erros Adequadamente: Implemente tratamento de erros específico para cada função
    
Teste os Decorators: Crie testes específicos para as validações
    
Considere Performance: Evite validações desnecessárias em funções críticas de performance

# Exemplo de Pipeline Completo com Decorators

In [ ]:
# Definindo schemas
schema_dados_brutos = pa.DataFrameSchema({
    'data': pa.Column('datetime64[ns]'),
    'produto': pa.Column(str),
    'quantidade': pa.Column(int, checks=pa.Check.gt(0)),
    'valor': pa.Column(float, checks=pa.Check.gt(0))
})

schema_dados_processados = pa.DataFrameSchema({
    'data': pa.Column('datetime64[ns]'),
    'produto': pa.Column(str),
    'quantidade': pa.Column(int, checks=pa.Check.gt(0)),
    'valor': pa.Column(float, checks=pa.Check.gt(0)),
    'valor_total': pa.Column(float, checks=pa.Check.gt(0))
})

schema_relatorio = pa.DataFrameSchema({
    'produto': pa.Column(str),
    'quantidade_total': pa.Column(int, checks=pa.Check.gt(0)),
    'valor_total': pa.Column(float, checks=pa.Check.gt(0)),
    'valor_medio': pa.Column(float, checks=pa.Check.gt(0))
})

# Pipeline de processamento
@pa.check_input(schema_dados_brutos)
@pa.check_output(schema_dados_processados)
def processar_dados(df):
    df['valor_total'] = df['quantidade'] * df['valor']
    return df

@pa.check_input(schema_dados_processados)
@pa.check_output(schema_relatorio)
def gerar_relatorio(df):
    relatorio = df.groupby('produto').agg({
        'quantidade': 'sum',
        'valor_total': 'sum'
    }).reset_index()
    
    relatorio['valor_medio'] = relatorio['valor_total'] / relatorio['quantidade']
    
    return relatorio

# Exemplo de uso
dados = pd.DataFrame({
    'data': pd.date_range(start='2023-01-01', periods=5),
    'produto': ['A', 'B', 'A', 'B', 'A'],
    'quantidade': [10, 20, 15, 25, 30],
    'valor': [100.0, 200.0, 150.0, 250.0, 300.0]
})

try:
    dados_processados = processar_dados(dados)
    relatorio_final = gerar_relatorio(dados_processados)
    print(relatorio_final)
except pa.errors.SchemaError as e:
    print(f"Erro na validação: {e}")